# (Appendix) Building an MCP Host · Client · Server with FastMCP

**MCP (Model Context Protocol)** is an open protocol that lets an LLM app (Host) connect to external tools and data (Server)
in a standard way. Its architecture has three layers: **Host → Client(s) → Server(s)**.

| Layer | Role | In this notebook |
|---|---|---|
| **Server** | Provides tools, resources, and prompts | `FastMCP` server + `@mcp.tool` |
| **Client** | Connects to a server to list and call tools | `fastmcp.Client` (in-memory transport) |
| **Host** | The LLM decides which tool to use and executes it via the Client | **NVIDIA build `deepseek-v4.1-flash`** |

[FastMCP](https://gofastmcp.com/getting-started/welcome) lets you quickly build MCP servers/clients in Python.
In particular, it supports an **in-memory transport** (passing the server object directly to the Client), so you can run the
entire Server↔Client↔Host flow inside a single notebook without separate processes or a network.

> 📦 Installation & run: [`env_guides/M02_4_fastmcp_eng.md`](env_guides/M02_4_fastmcp_eng.md) · shared tools are in [`agentic_lib/`](agentic_lib)

### Prerequisites
- `M02_1_local_llm_eng` completed (`NVIDIA_API_KEY` and `OPENROUTER_API_KEY` set in `.env`, OpenAI-compatible API connection verified)
- Package `fastmcp` (installed automatically by the cell below)

---
## 0. Environment Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # add notebooks/ to the import path

import utils
utils.reload_env()  # reload .env + print the current provider status

from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text  # provider-agnostic response normalization (removes <think>)

# Install FastMCP (passes quickly if already installed)
utils.uv_install(['fastmcp'])

llm = utils.get_llm()
print('Ready:', utils.LLM_PROVIDER)

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['fastmcp']


준비 완료: nvidia


---
## 1. MCP **Server** — `FastMCP` + `@mcp.tool`

Create a server with `FastMCP(name=...)`, and adding `@mcp.tool` to an ordinary Python function turns it into an MCP tool.
The function's **type hints and docstring** become the tool's input schema and description as is, and are exposed to the client/LLM.

In [2]:
import math, datetime
from fastmcp import FastMCP

# Create the MCP server
mcp = FastMCP(name="ToolServer", instructions="An MCP server that provides calculation, weather, and time tools")

# Safe math symbols exposed to eval (blocks arbitrary code execution)
_MATH = {k: getattr(math, k) for k in ['sqrt','pow','sin','cos','tan','log','log10','exp','pi','e','factorial','floor','ceil']}

@mcp.tool
def calculator(expression: str) -> str:
    """Evaluates a math expression. Examples: '2 ** 10', 'sqrt(144)', '(3+4)*5'."""
    try:
        # Convert the caret (^) that LLMs often use into Python exponentiation (**)
        return f"{expression} = {eval(expression.replace('^', '**'), {'__builtins__': {}}, _MATH)}"
    except Exception as e:
        return f"Error: {e}"

@mcp.tool
def get_weather(city: str) -> str:
    """Looks up the current weather for a city (simulated)."""
    return f"{city} weather: sunny, 22°C"

@mcp.tool
def get_current_time() -> str:
    """Returns the current date and time."""
    return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('MCP server created:', mcp.name)

MCP 서버 생성 완료: ToolServer


---
## 2. MCP **Client** — Connecting to the Server via In-Memory Transport

If you **pass the server object directly**, as in `Client(mcp)`, FastMCP connects in memory without a separate process or network
(ideal for testing and learning). The client has an `async` API, so manage the connection lifecycle with `async with` and
make calls with `await`. (Jupyter supports top-level `await` in cells.)

- `await client.list_tools()` — list the tools and schemas provided by the server
- `await client.call_tool(name, args)` — execute a tool → `CallToolResult` (`.data` structured result, `.content` blocks)

In [3]:
from fastmcp import Client

# In-memory transport: pass the server object directly to the Client
async with Client(mcp) as client:
    # 1) Tool list (discovery)
    tools = await client.list_tools()
    print('=== MCP tools provided by the server ===')
    for t in tools:
        params = list((t.inputSchema or {}).get('properties', {}).keys())
        print(f"  - {t.name}{tuple(params)}: {(t.description or '').splitlines()[0]}")

    # 2) Call tools directly
    print('\n=== Tool calls ===')
    r1 = await client.call_tool('calculator', {'expression': '2 ** 10'})
    print('  calculator ->', r1.data)          # .data: structured return value
    r2 = await client.call_tool('get_weather', {'city': 'Seoul'})
    print('  get_weather ->', r2.data)

=== 서버가 제공하는 MCP 도구 ===
  - calculator('expression',): 수학 수식을 계산합니다. 예: '2 ** 10', 'sqrt(144)', '(3+4)*5'.
  - get_weather('city',): 도시의 현재 날씨를 조회합니다(시뮬레이션).
  - get_current_time(): 현재 날짜와 시간을 반환합니다.

=== 도구 호출 ===
  calculator -> 2 ** 10 = 1024
  get_weather -> 서울 날씨: 맑음, 22도


---
## 3. MCP **Host** — The LLM Selects Tools and Executes Them via the Client

The Host is the LLM app. The flow is as follows.

```
User request → [Host] LLM looks at the MCP tool schemas and decides which tool to call (tool_calls)
             → [Client] calls the chosen tool on the MCP server (call_tool)
             → [Server] returns the tool execution result
             → [Host] feeds the result back to the LLM to generate the final natural-language answer
```

The MCP tool schemas (`inputSchema`) retrieved by the client are passed as is to the LLM's `bind_tools()`,
so the LLM selects tools via standard `tool_calls`.

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

async def mcp_host(question: str, max_turns: int = 8):
    """MCP Host loop: LLM tool selection → run the server tool via the Client → feed results back for the final answer.

    Depending on the provider, a single response may call several tools (in parallel) or one at a time (sequentially).
    Models that support only 'one tool at a time' (e.g. the retired NVIDIA llama-3.1-8b) raise a 500 error
    ('only supports single tool-calls') when multiple tool_calls appear in the conversation history, so we
    encourage single calls with bootstrap.bind_tools, keep only the first call with bootstrap.cap_tool_calls, and,
    if tools remain, loop to handle them sequentially (no effect on providers that support parallel calls).
    """
    async with Client(mcp) as client:
        # 1) Fetch the server tools and convert them to LLM schemas (OpenAI function format)
        mcp_tools = await client.list_tools()
        schemas = [{"type": "function",
                    "function": {"name": t.name, "description": t.description or "",
                                 "parameters": t.inputSchema}} for t in mcp_tools]
        # Absorb provider differences: encourage parallel_tool_calls=False for single-tool-call servers
        llm_with_tools = bootstrap.bind_tools(llm, schemas)

        messages = [SystemMessage(content="Use the provided MCP tools if needed and answer accurately. /no_think"),
                    HumanMessage(content=question)]
        print(f"[Question] {question}")

        # 2-4) Until there are no more tool calls: LLM decides → execute via Client → feed back the result
        resp = None
        for _ in range(max_turns):
            resp = llm_with_tools.invoke(messages)
            resp = bootstrap.cap_tool_calls(resp)   # keep only the first call for single-tool servers (otherwise original)
            messages.append(resp)
            if not resp.tool_calls:
                break                                # no more tools to call → final answer
            print(f"[Host→LLM decision] tool_calls: {[(tc['name'], tc['args']) for tc in resp.tool_calls]}")
            for tc in resp.tool_calls:
                result = await client.call_tool(tc['name'], tc['args'])
                out = result.data if result.data is not None else result.content
                print(f"[Client→Server] {tc['name']}({tc['args']}) → {out}")
                messages.append(ToolMessage(content=str(out), tool_call_id=tc['id']))

        print(f"\n[Final answer] {to_text(resp.content) if resp is not None else ''}")

await mcp_host("Calculate 2 to the 10th power and tell me the weather in Seoul. Also, what time is it now?")

[질문] 2의 10제곱을 계산하고 서울 날씨도 알려줘. 지금 몇 시인지도.


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024



[최종 답변] ;;


---
## 4. Summary

| Layer | Implementation | Key API |
|---|---|---|
| **Server** | `FastMCP(name=...)` + `@mcp.tool` | Function type hints/docstring → tool schema generated automatically |
| **Client** | `Client(mcp)` (in-memory) | `await list_tools()`, `await call_tool(name, args)` |
| **Host** | NVIDIA build `deepseek-v4.1-flash` | Tool schemas via `bind_tools` → `tool_calls` → executed via the Client |

### In-Memory vs. Real Transport
For learning and testing, this notebook used the **in-memory transport** (passing the server object directly). In a real deployment,
the server runs as an independent process and clients connect over a standard transport.

```python
# Run the server over stdio (separate process; e.g. server.py)
if __name__ == '__main__':
    mcp.run()                       # default stdio transport

# Connect from the client (auto-detected from the transport string/config)
# async with Client('server.py') as client:      # stdio
# async with Client('http://localhost:8000/mcp') as client:   # HTTP
```

- Thanks to the **Server/Client/Host separation**, the same Host (LLM) can combine multiple MCP servers (internal DBs, files, search, etc.) in a standard way.

### References
- FastMCP: https://gofastmcp.com/getting-started/welcome
- MCP specification: https://modelcontextprotocol.io